# KOAI / IOAI 2027 — Portfolio 01: NumPy Preprocessing

**Foundational learning project.** Implement and test mean centering, feature standardization, safe treatment of constant features, and reusable leakage-free fit/transform functions using NumPy.

**Tools:** Python, NumPy, Google Colab  
**Scope:** small illustrative datasets, not a trained predictive model or benchmark.

Run **Runtime → Run all** from a fresh Colab runtime to reproduce the results.

## Dataset and mathematical objective

The initial dataset represents three students (rows) and three numerical features (columns): height (cm), weight (kg), and study hours. For each feature $j$, mean-centering applies

$$X'_{ij} = X_{ij} - \mu_j$$

where $\mu_j$ is the mean of column $j$. Later sections add independent train/test datasets and avoid fitting the scaler on test data.

In [1]:
import numpy as np

X = np.array([
    [170, 65, 8],
    [180, 75, 6],
    [160, 55, 9]
])

column_means = X.mean(axis=0)
print("X shape:", X.shape)
print("Means shape:", column_means.shape)
print("Feature means:", column_means)

X_centered = X - column_means

print(X_centered)

centered_means = X_centered.mean(axis=0)

print("Means after centering:", centered_means)

test_passed = np.allclose(centered_means, 0)

print("Test passed:", test_passed)

assert np.allclose(
    X_centered.mean(axis=0), 0
)

print("Mean-centering test passed!")




X shape: (3, 3)
Means shape: (3,)
Feature means: [170.          65.           7.66666667]
[[  0.           0.           0.33333333]
 [ 10.          10.          -1.66666667]
 [-10.         -10.           1.33333333]]
Means after centering: [ 0.00000000e+00  0.00000000e+00 -2.96059473e-16]
Test passed: True
Mean-centering test passed!


## 1. Mean Centering

### Objective
Mean centering shifts each feature so its average becomes zero, while preserving the relative differences between data points.

### Mathematical Principle
To center a feature, calculate its average value, then subtract that average from every observation. The values above the average become positive, values below it become negative, and the new average is mathematically zero, although floating-point calculations may produce very small numerical errors. The spread and relative ordering of the data remain unchanged.



### Implementation

`X_centered = X - X.mean(axis=0)`

`X.mean(axis=0)` gives one mean per column. NumPy broadcasting automatically applies each column’s mean to every row, eliminating an explicit Python loop.

### Validation

`assert np.allclose(X_centered.mean(axis=0), 0.0)`

`np.allclose` checks that the centered means are zero within floating-point tolerance.



## 2. Feature Standardization
Standardization transforms each nonconstant feature to have approximately zero mean and a population standard deviation of one. Constant features have zero standard deviation and require special handling to avoid division by zero.

In [2]:

# Calculate the standard deviation of each feature
feature_stds = X.std(axis=0)

# Standardize each feature
X_standardized = (X - column_means) / feature_stds

print("Feature standard deviations:", feature_stds)
print("Standardized data:")
print(X_standardized)


Feature standard deviations: [8.16496581 8.16496581 1.24721913]
Standardized data:
[[ 0.          0.          0.26726124]
 [ 1.22474487  1.22474487 -1.33630621]
 [-1.22474487 -1.22474487  1.06904497]]


In [3]:

# Test 1: Each feature should have mean zero
assert np.allclose(
    X_standardized.mean(axis=0), 0
)

# Test 2: Each feature should have standard deviation one
assert np.allclose(
    X_standardized.std(axis=0), 1
)

print("Standardization tests passed!")


Standardization tests passed!


### Mathematical Principle

Standardization shifts each nonconstant feature to have mean approximately 0 and population standard deviation approximately 1. A constant feature requires a safe denominator and remains at standard deviation 0. It changes the center and scale of the data while preserving the relative ordering of values.


### Implementation

Use `axis=0` because it calculates statistics down the rows, separately for each column. This ensures each feature is standardized using its own mean and standard deviation.

### Validation

Check both the mean and standard deviation because standardization requires both: a mean of 0 confirms centering, and a standard deviation of 1 confirms scaling. Testing only one would not fully verify the transformation

## 3. Edge Case: Zero-Variance Features

### Objective

Investigate how constant features affect standardization
and develop a safe method to handle them.

In [4]:

# Add a constant feature (age)

ages = np.array([16, 16, 16])

X_extended = np.column_stack((X, ages))

print("Extended dataset:")
print(X_extended)

print("Shape:", X_extended.shape)

# Calculate feature statistics

extended_means = X_extended.mean(axis=0)
extended_stds = X_extended.std(axis=0)

print("Means:", extended_means)
print("Standard deviations:", extended_stds)

# Identify zero-variance features

zero_variance_mask = extended_stds == 0

print("Zero-variance mask:", zero_variance_mask)







Extended dataset:
[[170  65   8  16]
 [180  75   6  16]
 [160  55   9  16]]
Shape: (3, 4)
Means: [170.          65.           7.66666667  16.        ]
Standard deviations: [8.16496581 8.16496581 1.24721913 0.        ]
Zero-variance mask: [False False False  True]


In [5]:
safe_stds = np.where(zero_variance_mask, 1, extended_stds)

print("Safe standard deviations:", safe_stds)

X_extended_standardized = (X_extended - extended_means) / safe_stds

print("Safely standardized data:")
print(X_extended_standardized)

Safe standard deviations: [8.16496581 8.16496581 1.24721913 1.        ]
Safely standardized data:
[[ 0.          0.          0.26726124  0.        ]
 [ 1.22474487  1.22474487 -1.33630621  0.        ]
 [-1.22474487 -1.22474487  1.06904497  0.        ]]


In [6]:


# Test 1: All values must be finite
assert np.isfinite(X_extended_standardized).all()
print("Test 1 passed: All values finite")

# Test 2: All feature means should be approximately zero
assert np.allclose(
    X_extended_standardized.mean(axis=0), 0
)
print("Test 2 passed: Mean = 0")

# Test 3: Nonconstant features should have std = 1
assert np.allclose(
    X_extended_standardized.std(axis=0)[~zero_variance_mask],
    1
)
print("Test 3 passed: Nonconstant std = 1")

# Test 4: Constant features should become zero
assert np.allclose(
    X_extended_standardized[:, zero_variance_mask],
    0
)
print("Test 4 passed: Constant features = 0")



Test 1 passed: All values finite
Test 2 passed: Mean = 0
Test 3 passed: Nonconstant std = 1
Test 4 passed: Constant features = 0


### Handling Zero-Variance Features

#### Problem
Ordinary standardization fails for constant features because their standard deviation is zero, making the z-score denominator undefined and producing `NaN`.

#### Solution
`np.where(std == 0, 1.0, std)` replaces zero standard deviations with 1, creating a safe denominator. A constant feature then becomes all zeros instead of causing an error.

#### Validation
The four tests verify that all standardized values are finite, all feature means are approximately zero, nonconstant features have standard deviation approximately one, and constant features become zero. Together, these tests validate safe standardization on the current dataset.

## 4. Leakage-Free Train/Test Preprocessing

### Objective
Fit preprocessing statistics using training data
only, then apply the same transformation to
training and test data.

In [7]:

# Training data: Height (cm), Weight (kg)

X_train = np.array([
    [160, 60],
    [170, 70],
    [180, 80]
], dtype=float)

# Unseen test data
X_test = np.array([
    [200, 90],
    [210, 100]
], dtype=float)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Fit preprocessing statistics using training data ONLY
train_means = X_train.mean(axis=0)
train_stds = X_train.std(axis=0)

safe_train_stds = np.where(train_stds == 0, 1, train_stds)

print("Training means:", train_means)
print("Training stds:", train_stds)


Training shape: (3, 2)
Test shape: (2, 2)
Training means: [170.  70.]
Training stds: [8.16496581 8.16496581]


In [8]:

# Apply the training statistics to both datasets

X_train_scaled = (X_train - train_means) / safe_train_stds

X_test_scaled = (X_test - train_means) / safe_train_stds

print("Scaled training data:")
print(X_train_scaled)

print("Scaled test data:")
print(X_test_scaled)

print("Training feature means:", X_train_scaled.mean(axis=0))
print("Test feature means:", X_test_scaled.mean(axis=0))


Scaled training data:
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
Scaled test data:
[[3.67423461 2.44948974]
 [4.89897949 3.67423461]]
Training feature means: [0. 0.]
Test feature means: [4.28660705 3.06186218]


In [9]:

# Test 1: Scaled training features have mean 0
assert np.allclose(X_train_scaled.mean(axis=0), 0)

# Test 2: Scaled training features have std 1
assert np.allclose(X_train_scaled.std(axis=0), 1)

# Test 3: Dataset shapes are preserved
assert X_train_scaled.shape == X_train.shape
assert X_test_scaled.shape == X_test.shape

# Test 4: All outputs are finite
assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_test_scaled).all()

print("All train/test preprocessing tests passed!")


All train/test preprocessing tests passed!


### Preventing Data Leakage

#### Problem
Data leakage can occur when preprocessing statistics are calculated using validation or test data. This violates the held-out evaluation protocol and can make performance estimates unreliable.

#### Implementation
The means and standard deviations were calculated using **training data only**. The same fitted values were then reused to transform both training and test matrices.

#### Experimental Results
The standardized training means were `[0., 0.]`, while the test means were approximately `[4.287, 3.062]`. These test means are not required to be zero because test observations were transformed using the training distribution.

#### Validation
This section tests training mean and standard deviation, output shapes, and finite values. Later integrated tests also compare the test transformation with a calculation using the saved training statistics. The tests check specific properties but are not a comprehensive proof against every possible source of data leakage.

## 5. Reusable Standardization Functions

### Objective

Implement reusable NumPy functions for fitting
and applying a standardization transformation
without introducing data leakage.

In [10]:


def fit_standardizer(X_train):
    means = X_train.mean(axis=0)
    stds = X_train.std(axis=0)
    safe_stds = np.where(stds == 0, 1, stds)

    return means, safe_stds


# Call the function and unpack its outputs
fitted_means, fitted_stds = fit_standardizer(X_train)

print("Fitted means:", fitted_means)
print("Fitted standard deviations:", fitted_stds)


# Compare the function with our earlier calculations
assert np.allclose(fitted_means, train_means)
assert np.allclose(fitted_stds, safe_train_stds)

print("Function tests passed!")




Fitted means: [170.  70.]
Fitted standard deviations: [8.16496581 8.16496581]
Function tests passed!


In [11]:

def transform_standardizer(X, means, safe_stds):
    """
    Standardize a dataset using previously
    fitted means and standard deviations.
    """

    X_scaled = (X - means) / safe_stds

    return X_scaled

# Reuse the statistics fitted in Step 8A

train_scaled_v2 = transform_standardizer(
    X_train, fitted_means, fitted_stds
)

test_scaled_v2 = transform_standardizer(
    X_test, fitted_means, fitted_stds
)

print("Training data:")
print(train_scaled_v2)

print("Test data:")
print(test_scaled_v2)

Training data:
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
Test data:
[[3.67423461 2.44948974]
 [4.89897949 3.67423461]]


In [12]:

# Test 1: Training means
assert np.allclose(train_scaled_v2.mean(axis=0), 0)

# Test 2: Training standard deviations
assert np.allclose(train_scaled_v2.std(axis=0), 1)

# Test 3: Output shapes
assert train_scaled_v2.shape == X_train.shape
assert test_scaled_v2.shape == X_test.shape

# Test 4: Test data uses training statistics
expected_test = (X_test - fitted_means) / fitted_stds
assert np.allclose(test_scaled_v2, expected_test)

# Test 5: All outputs are finite
assert np.isfinite(train_scaled_v2).all()
assert np.isfinite(test_scaled_v2).all()

print("All 5 integrated tests passed!")


All 5 integrated tests passed!


In [13]:

# Add a constant age feature
X_train_constant = np.column_stack((
    X_train,
    np.array([16, 16, 16])
))

X_test_constant = np.column_stack((
    X_test,
    np.array([16, 16])
))

# Fit the reusable standardizer
means_c, stds_c = fit_standardizer(X_train_constant)

# Transform both datasets
train_c = transform_standardizer(
    X_train_constant, means_c, stds_c
)

test_c = transform_standardizer(
    X_test_constant, means_c, stds_c
)

# Check the constant feature
assert np.allclose(train_c[:, -1], 0)
assert np.allclose(test_c[:, -1], 0)

# Check there are no invalid values
assert np.isfinite(train_c).all()
assert np.isfinite(test_c).all()

print("Reusable functions handle constant features!")
print("Scaled training data:")
print(train_c)


Reusable functions handle constant features!
Scaled training data:
[[-1.22474487 -1.22474487  0.        ]
 [ 0.          0.          0.        ]
 [ 1.22474487  1.22474487  0.        ]]


### Reusable Fit and Transform Functions

#### Design
We separated fitting from transforming to prevent data leakage. `fit` learns statistics only from training data, while `transform` applies those learned statistics to both training and test data, keeping the test set unseen during preprocessing.

#### Implementation
`fit` receives the training feature matrix and returns the learned mean and a safe scaling denominator for each feature. `transform` receives a feature matrix plus the stored mean and safe scaling denominator, then returns the standardized matrix.

#### Testing
We validated that training features have mean approximately zero and standard deviation approximately one. We also confirmed that test data is transformed using training statistics, constant features are handled safely, and no invalid values such as `NaN` or infinity appear.

#### Limitations and Future Improvements
The current implementation assumes all inputs are numeric and does not handle missing values, categorical features, or outliers. Future improvements could include missing-value imputation, categorical encoding, robust scaling for outliers, and integration into a reusable preprocessing pipeline.

## Conclusion and next steps

On these small synthetic datasets, the NumPy transformations and assertions run successfully. The reusable functions fit statistics exclusively on the training data, handle constant training features without division by zero, and transform test samples with the saved parameters.

**Limitations:** numerical arrays only; no missing-value strategy, categorical encoding, robust outlier treatment, explicit input-shape validation, or automated model training. An extended project should compare with `sklearn.preprocessing.StandardScaler`, use a realistic dataset, and evaluate an actual model with a sound validation scheme.

**Project provenance:** developed as a guided coding and debugging exercise during KOAI / IOAI preparation.